### lib

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress TensorFlow C++ logs

import pandas as pd
import numpy as np
#for HAR&co methods
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression #per training
from sklearn.metrics import mean_squared_error, mean_absolute_error
#For ML methods
from sklearn.preprocessing import StandardScaler #per scaling

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, GRU #RNN
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Flatten #CNN

!pip install keras-tcn --quiet #TCN
from tcn import TCN

#DM test
from scipy import stats
import itertools
from statsmodels.stats.diagnostic import acorr_ljungbox
import matplotlib.pyplot as plt

##loading data


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
pathcrypto = '/content/drive/MyDrive/tesi_rv_forecast/CRYPTO' #percorso cartella
files= os.listdir(pathcrypto)

### btcusd


In [ ]:
#loading files
new_cols = ["dates", "LastPrices", "LastReturns", "RV", "RQ", "nRQ", "AbsRet", "BPV", "RSVp", "RSVn", "Returns"]

def load_crypto_df(file_name):
    file_path = os.path.join(pathcrypto, file_name)
    df = pd.read_csv(file_path, sep=";")
    df.columns = new_cols
    df['dates'] = pd.to_datetime(df['dates'])
    return df.set_index('dates')

btcusd = load_crypto_df("btcusd_df.csv")

#--------------------------------------------------------------------------------------------------
#CHECKS
def check_index(df):
    print('ckeck for ordered dates', df.index.is_monotonic_increasing)  # check for ordered data
    all_days = pd.date_range(df.index.min(), df.index.max(), freq="D") #check for missing days
    missing = all_days.difference(df.index)
    #print("Date mancanti:",missing)
    print('presence of Nan BEFORE',"\n",df.isna().sum(), "\n") #count missing rows

check_index(btcusd)

#------------------------------------------------------------------------------------------------------------------------------
#handling missing values
def smart_fill_drop(df, max_gap=5): #for less of 5gap rows it interpolates
    df_clean = btcusd.copy()
    for col in df_clean.columns:
        # Find consecutive NaN groups
        na_groups = df_clean[col].isna().astype(int).groupby(df_clean[col].notna().astype(int).cumsum()).cumsum()
        long_gaps = na_groups > max_gap
        df_clean[col] = df_clean[col].ffill().bfill() #forward fill
        df_clean.loc[long_gaps, col] = pd.NA # Drop rows where the gap was too long
    df_clean = df_clean.dropna()  # Drop rows with any remaining NaN
    return df_clean


btcusd_clean = smart_fill_drop(btcusd, max_gap=5)

btcusd_clean= btcusd_clean.reindex(columns=["RV","BPV", "RSVp", "RSVn", "RQ", "nRQ",
                                            "LastReturns", "Returns", "AbsRet", "LastPrices"])
btcusd_df=btcusd_clean.copy()

#print("Cleaned shape:", btceur_clean.shape)
#print('presence of missing values now',"\n" ,btceur_clean.isna().sum())
df=btcusd_df.copy()

In [ ]:
df.head()

### MALL

In [ ]:
# =========================
#feature engineering
# =========================

def feature_engineering(df):
    #HAR
    df['RV_lag1'] = df['RV'].shift(1)
    df['RV_lagW'] = df['RV'].rolling(window=7).mean().shift(1)   # media settimana precedente
    df['RV_lagM'] = df['RV'].rolling(window=30).mean().shift(1)  # media mese precedente
     #logHAR
    df['logRV'] = np.log(df['RV'])
    df['logRV_lag1'] = df['logRV'].shift(1)
    df['logRV_lagW'] = df['logRV'].rolling(window=7).mean().shift(1)
    df['logRV_lagM'] = df['logRV'].rolling(window=30).mean().shift(1)

     #LevHAR
    df['negRet'] = df['LastReturns'].apply(lambda x: x if x < 0 else 0) # LevHAR
    df['negRet_lag1'] = df['negRet'].shift(1) # negative returns
    df['negRet_lagW'] = df['negRet'].rolling(7).mean().shift(1) # negative returns
    df['negRet_lagM'] = df['negRet'].rolling(30).mean().shift(1) # negative returns

    # HAR-CJ
    df['C'] = df['BPV']         # C = BPV!!!!
    df['J'] = df['RV'] - df['BPV']  # J = RV-C
    df['C_lag1'] = df['C'].shift(1)
    df['C_lagW'] = df['C'].rolling(7).mean().shift(1)
    df['C_lagM'] = df['C'].rolling(30).mean().shift(1)
    df['J_lag1'] = df['J'].shift(1)
    df['J_lagW'] = df['J'].rolling(7).mean().shift(1)
    df['J_lagM'] = df['J'].rolling(30).mean().shift(1)

    #LevHAR-CJ

    #SHAR
    df['RSVp_lag1'] = df['RSVp'].shift(1) #RV positive del girono precedente
    df['RSVn_lag1'] = df['RSVn'].shift(1) #RV negativa

    # HAR-L
    df['RV_lag2M'] = df['RV'].rolling(60).mean().shift(1)
    df['RV_lag3M'] = df['RV'].rolling(90).mean().shift(1)
    df['RV_lag6M'] = df['RV'].rolling(180).mean().shift(1)

    # HAR-Q
    df['HARQ_term'] = df['RV'].shift(1) * np.sqrt(df['RQ'].shift(1)) # HARQ: RV_{t-1} * sqrt(RQ_{t-1})
    df['RQ_lag1'] = df['RQ'].shift(1)
    df['RQ_lagW'] = df['RQ'].rolling(7).mean().shift(1)
    df['RQ_lagM'] = df['RQ'].rolling(30).mean().shift(1)

    #HAR MIDAS

    MHAR = df[['RV',
               'RV_lag1', 'RV_lagW', 'RV_lagM']].dropna()
    MALL = df[['RV',
               'RV_lag1','RV_lagW','RV_lagM', #HAR
               'logRV_lag1',  'logRV_lagW','logRV_lagM',  #logHAR
               'negRet_lag1','negRet_lagW','negRet_lagM', #LevHAR
               'C_lag1', 'C_lagW', 'C_lagM', #HAR-CJ
               'J_lag1', 'J_lagW', 'J_lagM',
               'RSVp_lag1', 'RSVn_lag1', #SHAR
               'RV_lag2M', 'RV_lag3M', 'RV_lag6M', #HARL
               'HARQ_term','RQ_lag1','RQ_lagW', 'RQ_lagM'  #HARQ
               ]].dropna()

    return MALL

# ==================================================

#partitioning 70-10-20
def split_dataset(df):
        n = len(df)
        train_end = int(n * 0.7)
        val_end = int(n * 0.8)  # 70 + 10

        train = df.iloc[:train_end]
        val   = df.iloc[train_end:val_end]
        test  = df.iloc[val_end:]

        return train, val, test
 # ==================================================

MALL = feature_engineering(df.copy())
MALL_train, MALL_val, MALL_test= split_dataset(MALL)

traditonals

In [ ]:
# ==============================================================================
# Model architecture : HAR, logHAR, LevHAR, HARCJ, LevHARCJ, SHAR, HARL, HARQ
# ==============================================================================
def fit_HAR(df):
    X = df[['RV_lag1', 'RV_lagW', 'RV_lagM']]
    y = df['RV']
    X = sm.add_constant(X)  # β0
    model = sm.OLS(y, X).fit()
    return model

def fit_logHAR(df):
    X = df[['logRV_lag1', 'logRV_lagW', 'logRV_lagM']]
    y = df['RV']
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    return model

def fit_LevHAR(df):
    X = df[['RV_lag1','RV_lagW','RV_lagM',
            'negRet_lag1','negRet_lagW','negRet_lagM']] #adds negative returns
    y = df['RV']
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    return model

def fit_HARCJ(df):
    X = df[['C_lag1', 'C_lagW', 'C_lagM',
            'J_lag1', 'J_lagW', 'J_lagM']]
    y = df['RV']
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    return model

def fit_LevHARCJ(df): #ibrido tra levHAR e HAR-CJ
    X = df[['C_lag1', 'C_lagW', 'C_lagM',
            'J_lag1', 'J_lagW', 'J_lagM',
            'negRet_lag1','negRet_lagW','negRet_lagM']]
    y = df['RV']
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    return model

def fit_SHAR(df):
    X = df[['RSVp_lag1','RSVn_lag1','RV_lagW','RV_lagM']]
    y = df['RV']
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    return model

def fit_HARL(df):
    X = df[['RV_lag1', 'RV_lagW', 'RV_lagM',
        'RV_lag2M', 'RV_lag3M', 'RV_lag6M']]
    y = df['RV']
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    return model

def fit_HARQ(df):
    X = df[['HARQ_term', 'RQ_lag1', 'RQ_lagW', 'RQ_lagM']]
    y = df['RV']
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    return model

In [ ]:
# ======================================================
# fit sul training
# ======================================================
har_model     = fit_HAR(MALL_train)
loghar_model=fit_logHAR(MALL_train)
levhar_model=fit_LevHAR(MALL_train)
harcj_model  = fit_HARCJ(MALL_train)
levharcj_model=fit_LevHARCJ(MALL_train)
shar_model   = fit_SHAR(MALL_train)
harl_model   = fit_HARL(MALL_train)
harq_model   = fit_HARQ(MALL_train)


def validate_model(model, df_train, df_val, regressors, target="RV"):
    # errori sul training
    X_train = sm.add_constant(df_train[regressors])
    y_train = df_train[target]
    y_train_pred = model.predict(X_train)

    mse_train  = mean_squared_error(y_train, y_train_pred)
    mae_train  = mean_absolute_error(y_train, y_train_pred)

    # errori sul validation
    X_val = sm.add_constant(df_val[regressors])
    y_val = df_val[target]
    y_val_pred = model.predict(X_val)

    mse_val  = mean_squared_error(y_val, y_val_pred)
    mae_val  = mean_absolute_error(y_val, y_val_pred)

    return {
        "MSE_train": mse_train, "MAE_train": mae_train,
        "MSE_val": mse_val, "MAE_val": mae_val
    }


#regressors
regressorsHAR     = ['RV_lag1','RV_lagW','RV_lagM']
regressorslogHAR  = ['logRV_lag1','logRV_lagW','logRV_lagM']
regressorsLevHAR  = ['RV_lag1','RV_lagW','RV_lagM',
                     'negRet_lag1','negRet_lagW','negRet_lagM']
regressorsHARCJ   = ['C_lag1','C_lagW','C_lagM',
                     'J_lag1','J_lagW','J_lagM']
regressorsLevHARCJ = ['C_lag1','C_lagW','C_lagM',
                      'J_lag1','J_lagW','J_lagM',
                      'negRet_lag1','negRet_lagW','negRet_lagM']
regressorsSHAR    = ['RSVp_lag1','RSVn_lag1','RV_lagW','RV_lagM']
regressorsHARL    = ['RV_lag1','RV_lagW','RV_lagM',
                     'RV_lag2M','RV_lag3M','RV_lag6M']
regressorsHARQ    = ['HARQ_term','RQ_lag1','RQ_lagW','RQ_lagM']

#==========================================
# Validation
#===========================================


metrics_har      = validate_model(har_model, MALL_train, MALL_val, regressorsHAR)
metrics_loghar=    validate_model(loghar_model, MALL_train, MALL_val, regressorslogHAR)
metrics_levhar   = validate_model(levhar_model, MALL_train, MALL_val, regressorsLevHAR)
metrics_harcj    = validate_model(harcj_model, MALL_train, MALL_val, regressorsHARCJ)
metrics_levharcj = validate_model(levharcj_model, MALL_train, MALL_val, regressorsLevHARCJ)
metrics_shar     = validate_model(shar_model, MALL_train, MALL_val, regressorsSHAR)
metrics_harl     = validate_model(harl_model, MALL_train, MALL_val, regressorsHARL)
metrics_harq     = validate_model(harq_model, MALL_train, MALL_val, regressorsHARQ)

# ===============================================================================================
# testing su MALL_test
# ======================================================================================

y_pred_HAR = har_model.predict(sm.add_constant(MALL_test[regressorsHAR]))
y_pred_logHAR = loghar_model.predict(sm.add_constant(MALL_test[regressorslogHAR]))
y_pred_LevHAR = levhar_model.predict(sm.add_constant(MALL_test[regressorsLevHAR]))
y_pred_HARCJ = harcj_model.predict(sm.add_constant(MALL_test[regressorsHARCJ]))
y_pred_LevHARCJ = levharcj_model.predict(sm.add_constant(MALL_test[regressorsLevHARCJ]))
y_pred_SHAR = shar_model.predict(sm.add_constant(MALL_test[regressorsSHAR]))
y_pred_HARL = harl_model.predict(sm.add_constant(MALL_test[regressorsHARL]))
y_pred_HARQ = harq_model.predict(sm.add_constant(MALL_test[regressorsHARQ]))

## NN X_train

In [ ]:
# ======================================================
# partitioning + scaling
# ======================================================
df_train, df_val, df_test= split_dataset(df)
features = df.columns.tolist() #features
target = 'RV' #target
scaler = StandardScaler()
scaler.fit(df[features]) #standardizzazione solo su train

# Trasforma train, val e test in dataframe
train_scaled = pd.DataFrame(scaler.transform(df_train[features]), columns=features, index=df_train.index)
val_scaled   = pd.DataFrame(scaler.transform(df_val[features]), columns=features, index=df_val.index)
test_scaled  = pd.DataFrame(scaler.transform(df_test[features]),    columns=features, index=df_test.index)

# ======================================================
# windowing
# ======================================================
def create_multivariate_sequences(df, target_col='RV', window=20):
    X, y = [], []
    data = df.values
    target_idx = df.columns.get_loc(target_col)

    for i in range(len(data) - window):
        X.append(data[i:i+window, :])            # finestra con tutte le features
        y.append(data[i+window, target_idx])     # target = RV del giorno successivo
    return np.array(X), np.array(y)

#creation of window
seq_len = 20

X_train, y_train = create_multivariate_sequences(train_scaled, target_col=target, window=seq_len)
X_val, y_val     = create_multivariate_sequences(val_scaled,   target_col=target, window=seq_len)
X_test, y_test   = create_multivariate_sequences(test_scaled,  target_col=target, window=seq_len)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


## fit RNN CNN

In [ ]:
seq_len = X_train.shape[1]
num_features = X_train.shape[2]

# ======================================================
#   LSTM base, LSTM robust,  GRU base , GRU robust
# ======================================================
def fit_LSTM(input_shape):
    model = Sequential([ LSTM(64, activation='tanh', input_shape=input_shape),
                         Dense(1) ])
    model.compile(optimizer='adam', loss='mse')
    return model

def fit_LSTM_robust(input_shape):
    model = Sequential([LSTM(64, return_sequences=True, input_shape=input_shape),
                        Dropout(0.2),
                        LSTM(32),
                        Dropout(0.2),
                        Dense(1)])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

def fit_GRU(input_shape):
    model = Sequential([ GRU(64, activation='tanh', input_shape=input_shape),
                          Dense(1)])
    model.compile(optimizer='adam', loss='mse')
    return model

def fit_GRU_robust(input_shape):
    model = Sequential([
        GRU(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        GRU(32),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

# ======================================================
# CNN-1D base, CNN-1D robust , TCN
# ======================================================

def fit_CNN1D(input_shape):
    model = Sequential([
        Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(50, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def fit_CNN1D_robust(input_shape):
    model = Sequential([
        Conv1D(filters=128, kernel_size=3, activation='relu', input_shape=input_shape),
        Conv1D(filters=64, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Flatten(),
        Dense(100, activation='relu'),
        Dropout(0.3),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

def fit_TCN(input_shape):
    model = Sequential([
        TCN(input_shape=input_shape,
            nb_filters=64,
            kernel_size=3,
            dilations=[1,2,4,8,16],
            dropout_rate=0.2,
            return_sequences=False),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


In [ ]:
# ================================================
# to prevent overfitting
# ================================================
early_stop = EarlyStopping(
                            monitor='val_loss',
                            patience=8,               # quante epoche attendere prima di fermarsi
                            restore_best_weights=True) # ripristina i pesi migliori

reduce_lr = ReduceLROnPlateau(
                              monitor='val_loss',
                              factor=0.5,   # riduci il learning rate
                              patience=4,
                              min_lr=1e-5)

callbacks = [early_stop, reduce_lr]

# ================================================
# training function and evaluation
# ================================================
def train_and_evaluate(model_fn, name, input_shape,
                       X_train, y_train, X_val, y_val,
                       X_test, y_test,
                       epochs=100, batch_size=32):

    print(f"\n===== Training {name} =====")
    model = model_fn(input_shape)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )

    # Evaluation
    val_loss = model.evaluate(X_val, y_val, verbose=0)
    test_loss = model.evaluate(X_test, y_test, verbose=0)

    print(f"{name} - Val MSE: {val_loss:.6f} | Test MSE: {test_loss:.6f}")

    # Predizioni finali sul test
    y_pred = model.predict(X_test).ravel()

    return model, history, y_pred



In [ ]:
input_shape = (X_train.shape[1], X_train.shape[2])


#LSTM e GRU
lstm_base, hist_lstm_base, y_pred_lstm_base = train_and_evaluate(
    fit_LSTM, "LSTM base", input_shape,
    X_train, y_train, X_val, y_val, X_test, y_test
)

lstm_robust, hist_lstm_robust, y_pred_lstm_robust = train_and_evaluate(
    fit_LSTM_robust, "LSTM robust", input_shape,
    X_train, y_train, X_val, y_val, X_test, y_test
)

gru_base, hist_gru_base, y_pred_gru_base = train_and_evaluate(
    fit_GRU, "GRU base", input_shape,
    X_train, y_train, X_val, y_val, X_test, y_test
)

gru_robust, hist_gru_robust, y_pred_gru_robust = train_and_evaluate(
    fit_GRU_robust, "GRU robust", input_shape,
    X_train, y_train, X_val, y_val, X_test, y_test
)




In [ ]:
import matplotlib.pyplot as plt

plt.plot(hist_lstm_base.history['loss'], label='Train Loss')
plt.plot(hist_lstm_base.history['val_loss'], label='Val Loss')
plt.legend()
plt.title("LSTM base - Learning curve")
plt.show()


In [ ]:

#CNN-1D ----- TCN------

cnn_base, hist_cnn_base, y_pred_cnn_base = train_and_evaluate(
    fit_CNN1D, "CNN-1D base", input_shape,
    X_train, y_train, X_val, y_val, X_test, y_test
)

cnn_robust, hist_cnn_robust, y_pred_cnn_robust = train_and_evaluate(
    fit_CNN1D_robust, "CNN-1D robust", input_shape,
    X_train, y_train, X_val, y_val, X_test, y_test
)

tcn, hist_tcn, y_pred_tcn = train_and_evaluate(
    fit_TCN, "TCN", input_shape,
    X_train, y_train, X_val, y_val, X_test, y_test
)

## DM test e testing??

In [ ]:
#errori MSE MAE QLIKE : LOSS FUNCTIONS
def mse(y_test, y_pred): #penalizza molto gli outlier (quadrato degli errori).
    return np.mean((y_test - y_pred)**2)
'''
def mae(y_test, y_pred): #MAE Più robusto agli outlier.
    return np.mean(np.abs(y_test - y_pred))

def qlike(y_test, y_pred): #Quasi-likelihood loss: e i valori previsti (y_pred) sono troppo vicini a zero, puoi avere divisioni strane o NaN.
    y_pred_safe = np.maximum(y_pred, 1e-8)  # evita valori QUASI VICINI A ZERO
    return np.mean((y_test / y_pred_safe) - np.log(y_test / y_pred_safe) - 1)
'''
# =============================
# Diebold-Mariano Test
# =============================
#DM TEST
def diebold_mariano_test(y_true, y_pred1, y_pred2, h=1, crit="MSE"): #CONFRONTA DUE PROBLEMI DUE A DUE
    """
    y_pred1, y_pred2: previsioni dei due modelli
    h: orizzonte (default 1)
    crit: "MSE", "MAE", "QLIKE"
    """
    e1 = (y_true - y_pred1)** 2 #errore modello 1
    e2 = (y_true - y_pred2)**2 #errore modello 2
    d= e1- e2

    mean_d = np.mean(d)
    var_d = np.var(d, ddof=1)

    # Newey-West correction (lag=h-1)
    nw_var = var_d
    for lag in range(1, h):
        gamma = np.cov(d[:-lag], d[lag:])[0, 1]
        nw_var += 2 * (1 - lag/h) * gamma

    DM_stat = mean_d / np.sqrt(nw_var / len(d))
    #p_value = 2 * (1 - stats.norm.cdf(np.abs(DM_stat)))
    return DM_stat

# =============================
# P
# =============================

def evaluate_models(df, models, target_col="RV"):

    y_true = df[target_col].values

    # 1. Metriche aggregate
    metrics = []
    for m in models:
        y_pred = df[m].values
        metrics.append([ m, mse(y_true, y_pred)])

    #metrics_df = pd.DataFrame(metrics, columns=["Model", "MSE", "MAE", "QLIKE"])
    metrics_df = pd.DataFrame(metrics, columns=["Model", "MSE"])

    # 2. DM test pairwise
    results = []
    for m1, m2 in itertools.combinations(models, 2):
        #for crit in ["MSE", "QLIKE"]:
        for crit in ["MSE"]:
            DM_stat = diebold_mariano_test(y_true, df[m1].values, df[m2].values, crit=crit)
            results.append([m1, m2, "MSE", DM_stat])
    dm_df = pd.DataFrame(results, columns=["Model 1", "Model 2", "Loss", "DM statistic"])

    return metrics_df, dm_df

# =============================# =============================# =============================



In [ ]:
print("y_test:", len(y_test))
for name, arr in [
    ("HAR", y_pred_HAR),
    ("logHAR", y_pred_logHAR),
    ("LevHAR", y_pred_LevHAR),
    ("HARCJ", y_pred_HARCJ),
    ("LevHARCJ", y_pred_LevHARCJ),
    ("SHAR", y_pred_SHAR),
    ("HARL", y_pred_HARL),
    ("HARQ", y_pred_HARQ),
    ("LSTM base", y_pred_lstm_base),
    ("LSTM robust", y_pred_lstm_robust),
    ("GRU base", y_pred_gru_base),
    ("GRU robust", y_pred_gru_robust),
    ("CNN-1D base", y_pred_cnn_base),
    ("CNN-1D robust", y_pred_cnn_robust),
    ("TCN", y_pred_tcn)
]:
    print(name, len(arr))


y_test e i modelli deep learning (LSTM, GRU, CNN, TCN) → 382 rows

HAR e varianti (HAR, logHAR, LevHAR, HARCJ, LevHARCJ, SHAR, HARL, HARQ) → 356 rows

Comparing to NNs models, HAR models have only 356 rows due to their lagged values, thus dropping initial rows.

Otherwise NNs keep all the 382 rows (through padding or sliding window)

To compare HARs and NNs models, it is necessary that y_test is tha same for all the models, so the y_test must be cut to 356 obs.

In [ ]:
# Allinere alla lunghezza minima
min_len = min(len(y_test), len(y_pred_HAR))  # = 356

forecast_errors = pd.DataFrame({
    "RV": y_test[-min_len:],   # prendi le ultime 356 righe
    "HAR": y_pred_HAR,
    "logHAR": y_pred_logHAR,
    "LevHAR": y_pred_LevHAR,
    "HARCJ": y_pred_HARCJ,
    "LevHARCJ": y_pred_LevHARCJ,
    "SHAR": y_pred_SHAR,
    "HARL": y_pred_HARL,
    "HARQ": y_pred_HARQ,
    "LSTM base": y_pred_lstm_base[-min_len:],
    "LSTM robust": y_pred_lstm_robust[-min_len:],
    "GRU base": y_pred_gru_base[-min_len:],
    "GRU robust": y_pred_gru_robust[-min_len:],
    "CNN-1D base": y_pred_cnn_base[-min_len:],
    "CNN-1D robust": y_pred_cnn_robust[-min_len:],
    "TCN": y_pred_tcn[-min_len:]
})


In [ ]:
models = ["HAR","logHAR","LevHAR","HARCJ", "LevHARCJ","SHAR","HARL","HARQ",
          "LSTM base","LSTM robust", "GRU base","GRU robust","CNN-1D base", "CNN-1D robust","TCN"]
#metrics_df, dm_df = evaluate_models_strict_noqlike(forecast_errors, models)
metrics_df, dm_df = evaluate_models(forecast_errors, models)


print(metrics_df)
print('*********************************************************************')
print(dm_df)

QLIKE issue

   * QLIKE requires only **strictly positive forecasts**, altrimenti va in errore.
   
   è **NaN per tutti i modelli** → significa che il calcolo è saltato per via di valori negativi o zero nelle previsioni (`y_pred`).

   * Tutti i DM test basati su QLIKE sono **NaN** → stesso problema: divisione per zero o log di valori non validi.

* I modelli **HAR e varianti** sono regressioni lineari → non garantiscono previsioni positive di volatilità → alcune stime sono 0 o negative.
* QLIKE richiede



* I tuoi NaN derivano da forecast negativi/zero → non validi per QLIKE.

* Con la correzione `np.maximum(y_pred, 1e-8)` elimini warning e ottieni QLIKE finite.
* Così potrai confrontare i modelli anche con QLIKE, non solo con MSE.




In [ ]:
def build_relmse_dm_table(df, models, target_col="RV"):
    y_true = df[target_col].values
    mse_values = {}

    # calcolo MSE per ciascun modello
    for m in models:
        y_pred = df[m].values
        mse_values[m] = np.mean((y_true - y_pred)**2)

    # inizializza dataframe per la tabella
    relmse_matrix = pd.DataFrame(np.ones((len(models), len(models))),
                                 index=models, columns=models)

    # DM test matrix
    dm_matrix = pd.DataFrame(np.zeros((len(models), len(models))),
                             index=models, columns=models)

    # riempi matrici
    for m1, m2 in itertools.combinations(models, 2):
        rel = mse_values[m1] / mse_values[m2]
        relmse_matrix.loc[m1, m2] = rel
        relmse_matrix.loc[m2, m1] = 1/rel

        # DM statistic
        DM_stat = diebold_mariano_test(y_true, df[m1].values, df[m2].values, crit="MSE")
        dm_matrix.loc[m1, m2] = round(DM_stat, 2)
        dm_matrix.loc[m2, m1] = round(-DM_stat,2)

    return mse_values, relmse_matrix, dm_matrix

mse_values, relmse_matrix, dm_matrix=build_relmse_dm_table(forecast_errors, models)

In [ ]:
#display(relmse_matrix)   # tabella MSE relativi

### dm_matrix

In [ ]:
display(dm_matrix)       # tabella DM statistics
dm_matrix.to_csv('dm_matrix_btcusd.csv')

In [ ]:
display(dm_df.head(50))